# MiniTienda - Registro y análisis de ventas
**Grupo 1** — Valencia, Loor, Cambera
Logica de Programacion — UIDE

Este notebook contiene el código ejecutable del desafío *MiniTienda* y celdas de prueba
que demuestran el cumplimiento de cada requisito:

1. Catálogo (tuplas) + precios/stock (diccionarios)
2. Registro de ventas (listas + pandas DataFrame)
3. Guardado/lectura desde CSV (archivos)
4. Métricas con NumPy
5. Gráfica de ingresos por producto con Matplotlib
6. Menú con bucle `while` y control de flujo completo (Retos A, B, C, D incluidos)


## 1. Importaciones

In [1]:
import os
from datetime import datetime, timedelta
import random

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


## 2. Catálogo (tuplas) y diccionarios de precios/stock
Cada producto del catálogo es una **tupla** `(id, nombre, categoria)`. El catálogo
en sí se guarda en una lista para poder agregar nuevos productos (Reto A), pero cada
registro individual mantiene su naturaleza de tupla.

In [2]:
CATALOGO = [
    (1, "Laptop", "Electronica"),
    (2, "Mouse", "Electronica"),
    (3, "Teclado", "Electronica"),
    (4, "Monitor", "Electronica"),
    (5, "Silla Gamer", "Muebles"),
    (6, "Audifonos", "Electronica"),
]

PRECIOS = {1: 850.00, 2: 15.50, 3: 25.00, 4: 199.99, 5: 120.00, 6: 45.00}
STOCK = {1: 10, 2: 50, 3: 40, 4: 15, 5: 8, 6: 25}

VENTAS_BUFFER = []   # lista de diccionarios -> luego se convierte en DataFrame
IDS_VENDIDOS = []    # lista/arreglo de ids vendidos

BASE_DIR = os.getcwd()
CSV_PATH = os.path.join(BASE_DIR, "ventas.csv")
LOG_PATH = os.path.join(BASE_DIR, "log.txt")
PNG_PATH = os.path.join(BASE_DIR, "ingresos.png")
COLUMNAS_CSV = ["id_venta", "fecha", "producto_id", "producto", "cantidad",
                "precio_unitario", "descuento", "total"]


## 3. Funciones modulares

In [3]:
def escribir_log(mensaje):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {mensaje}\n")

def buscar_producto(producto_id):
    for producto in CATALOGO:
        if producto[0] == producto_id:
            return producto
    return None

def mostrar_catalogo():
    print("\n--- CATALOGO DE PRODUCTOS ---")
    print(f"{'ID':<4}{'Nombre':<15}{'Categoria':<15}{'Precio':<10}{'Stock':<6}")
    for producto in CATALOGO:
        pid, nombre, categoria = producto
        precio = PRECIOS.get(pid, 0)
        stock = STOCK.get(pid, 0)
        print(f"{pid:<4}{nombre:<15}{categoria:<15}${precio:<9.2f}{stock:<6}")

def calcular_descuento(cantidad, subtotal):
    """Reto C: 5% de descuento si unidades >= 10."""
    if cantidad >= 10:
        return round(subtotal * 0.05, 2)
    return 0.0


In [4]:
def registrar_venta(producto_id=None, cantidad=None, interactivo=True):
    """
    Registra una venta. Si interactivo=True pide datos por consola (input),
    si es False usa los parametros producto_id/cantidad (para pruebas y demo).
    Usa try/except/else/finally y controla producto inexistente (Reto D -> log.txt)
    y stock insuficiente.
    """
    if interactivo:
        mostrar_catalogo()
        try:
            producto_id = int(input("\nIngrese el ID del producto a vender: "))
            cantidad = int(input("Ingrese la cantidad: "))
        except ValueError:
            print(" Entrada invalida: debe ingresar numeros enteros.")
            escribir_log("ERROR: entrada no numerica al registrar venta.")
            return
    else:
        try:
            producto_id = int(producto_id)
            cantidad = int(cantidad)
        except (TypeError, ValueError):
            print(" Entrada invalida en modo demo.")
            return

    try:
        producto = buscar_producto(producto_id)

        if producto is None:
            print(" El producto no existe en el catalogo.")
            escribir_log(f"INTENTO FALLIDO: producto_id={producto_id} no existe en el catalogo.")
            return

        if cantidad <= 0:
            print(" La cantidad debe ser mayor a 0.")
            escribir_log(f"INTENTO FALLIDO: cantidad invalida ({cantidad}) para producto_id={producto_id}.")
            return

        if cantidad > STOCK.get(producto_id, 0):
            print(f" Stock insuficiente. Stock disponible: {STOCK.get(producto_id, 0)}")
            escribir_log(f"INTENTO FALLIDO: stock insuficiente para producto_id={producto_id}.")
            return
    except Exception as e:
        print(f" Error inesperado: {e}")
        return
    else:
        precio_unitario = PRECIOS[producto_id]
        subtotal = precio_unitario * cantidad
        try:
            precio_promedio_unidad = subtotal / cantidad if cantidad != 0 else 0
        except ZeroDivisionError:
            print(" Division por cero controlada.")
            precio_promedio_unidad = 0
        finally:
            pass

        descuento = calcular_descuento(cantidad, subtotal)
        total = round(subtotal - descuento, 2)
        STOCK[producto_id] -= cantidad

        venta = {
            "id_venta": len(VENTAS_BUFFER) + 1,
            "fecha": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "producto_id": producto_id,
            "producto": producto[1],
            "cantidad": cantidad,
            "precio_unitario": precio_unitario,
            "descuento": descuento,
            "total": total,
        }
        VENTAS_BUFFER.append(venta)
        IDS_VENDIDOS.append(producto_id)
        print(f" Venta registrada: {cantidad}x {producto[1]} -> Total: ${total:.2f}"
              f"{' (con descuento 5%)' if descuento > 0 else ''}")
        escribir_log(f"VENTA OK: producto_id={producto_id}, cantidad={cantidad}, total={total}")


In [5]:
def guardar_ventas_csv():
    """Guarda (append) las ventas del buffer en ventas.csv usando pandas."""
    if not VENTAS_BUFFER:
        print(" No hay ventas nuevas para guardar.")
        return
    df_nuevo = pd.DataFrame(VENTAS_BUFFER, columns=COLUMNAS_CSV)
    try:
        if os.path.exists(CSV_PATH):
            df_existente = pd.read_csv(CSV_PATH)
            df_nuevo["id_venta"] = range(len(df_existente) + 1, len(df_existente) + 1 + len(df_nuevo))
            df_final = pd.concat([df_existente, df_nuevo], ignore_index=True)
        else:
            df_final = df_nuevo
        df_final.to_csv(CSV_PATH, index=False)
        print(f" {len(df_nuevo)} venta(s) guardadas en {CSV_PATH}")
        escribir_log(f"CSV actualizado con {len(df_nuevo)} venta(s) nuevas.")
        VENTAS_BUFFER.clear()
    except Exception as e:
        print(f" Error al guardar el CSV: {e}")
        escribir_log(f"ERROR al guardar CSV: {e}")

def leer_ventas_csv():
    """Lee ventas.csv; controla archivo inexistente."""
    try:
        df = pd.read_csv(CSV_PATH)
    except FileNotFoundError:
        print(" El archivo ventas.csv no existe todavia.")
        escribir_log("ERROR: intento de leer ventas.csv pero no existe.")
        return None
    else:
        print(f"\n--- VENTAS REGISTRADAS ({len(df)}) ---")
        print(df.to_string(index=False))
        return df
    finally:
        pass


In [6]:
def calcular_metricas():
    """Calcula metricas con NumPy: mean, std, sum sobre el arreglo de totales."""
    df = leer_ventas_csv()
    if df is None or df.empty:
        print(" No hay datos para calcular metricas.")
        return
    totales = np.array(df["total"])
    print("\n--- METRICAS (NumPy) ---")
    print(f"Suma total de ingresos : ${np.sum(totales):.2f}")
    print(f"Promedio por venta      : ${np.mean(totales):.2f}")
    print(f"Desviacion estandar     : ${np.std(totales):.2f}")
    print(f"Venta maxima            : ${np.max(totales):.2f}")
    print(f"Venta minima            : ${np.min(totales):.2f}")

def graficar_ingresos(guardar_png=False):
    """Agrupa ingresos por producto con pandas (groupby) y grafica con matplotlib."""
    df = leer_ventas_csv()
    if df is None or df.empty:
        print(" No hay datos para graficar.")
        return
    ingresos_por_producto = df.groupby("producto")["total"].sum().sort_values(ascending=False)
    plt.figure(figsize=(9, 5))
    ingresos_por_producto.plot(kind="bar", color="#4C72B0")
    plt.title("Ingresos por producto - MiniTienda")
    plt.xlabel("Producto")
    plt.ylabel("Ingresos ($)")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    if guardar_png:
        plt.savefig(PNG_PATH)
        print(f" Grafico exportado como {PNG_PATH}")
        escribir_log("Grafico de ingresos exportado a PNG.")
    plt.show()


In [7]:
def agregar_producto(nombre, categoria, precio, stock_inicial):
    """Reto A: agrega un producto nuevo al catalogo y a precios/stock."""
    nuevo_id = max(p[0] for p in CATALOGO) + 1
    CATALOGO.append((nuevo_id, nombre, categoria))
    PRECIOS[nuevo_id] = precio
    STOCK[nuevo_id] = stock_inicial
    print(f" Producto '{nombre}' agregado con ID {nuevo_id}.")
    escribir_log(f"Producto nuevo agregado: id={nuevo_id}, nombre={nombre}, precio={precio}, stock={stock_inicial}")
    return nuevo_id

def actualizar_precio_stock(producto_id, nuevo_precio=None, nuevo_stock=None):
    """Actualiza precio y/o stock de un producto existente."""
    producto = buscar_producto(producto_id)
    if producto is None:
        print(" Producto no encontrado.")
        escribir_log(f"INTENTO FALLIDO: actualizar producto inexistente id={producto_id}.")
        return
    if nuevo_precio is not None:
        PRECIOS[producto_id] = float(nuevo_precio)
    if nuevo_stock is not None:
        STOCK[producto_id] = int(nuevo_stock)
    print(" Producto actualizado correctamente.")
    escribir_log(f"Producto actualizado: id={producto_id}, precio={PRECIOS[producto_id]}, stock={STOCK[producto_id]}")


## 4. Menú principal (bucle `while` + control de flujo)
La función `main()` reproduce el menú de consola completo. En este notebook se
muestra el código; para usarlo de forma interactiva ejecútese `main()` en una
terminal o en Jupyter con `input()` habilitado.

In [8]:
def mostrar_menu():
    print("\n===== MINITIENDA - MENU PRINCIPAL =====")
    print("1) Ver catalogo")
    print("2) Registrar venta")
    print("3) Guardar ventas en CSV")
    print("4) Ver ventas y metricas (NumPy)")
    print("5) Graficar ingresos por producto")
    print("6) Exportar grafico a PNG")
    print("7) Agregar producto nuevo (Reto A)")
    print("8) Actualizar precio/stock de un producto")
    print("0) Salir")

def main():
    intentos_invalidos = 0
    while True:
        mostrar_menu()
        opcion = input("Seleccione una opcion: ").strip()
        try:
            if opcion == "0":
                print("Guardando datos antes de salir...")
                guardar_ventas_csv()
                print("Hasta luego!")
                break
            elif opcion == "1":
                mostrar_catalogo()
            elif opcion == "2":
                registrar_venta(interactivo=True)
            elif opcion == "3":
                guardar_ventas_csv()
            elif opcion == "4":
                calcular_metricas()
            elif opcion == "5":
                graficar_ingresos(guardar_png=False)
            elif opcion == "6":
                graficar_ingresos(guardar_png=True)
            elif opcion == "7":
                nombre = input("Nombre: ")
                categoria = input("Categoria: ")
                precio = float(input("Precio: "))
                stock_inicial = int(input("Stock inicial: "))
                agregar_producto(nombre, categoria, precio, stock_inicial)
            elif opcion == "8":
                pid = int(input("ID producto: "))
                actualizar_precio_stock(pid)
            else:
                intentos_invalidos += 1
                print(" Opcion invalida, intente de nuevo.")
                if intentos_invalidos >= 3:
                    print(" Demasiados intentos invalidos.")
                    intentos_invalidos = 0
                continue
        except KeyboardInterrupt:
            print("\nInterrumpido. Guardando y saliendo...")
            guardar_ventas_csv()
            break
        except Exception as e:
            print(f" Error inesperado en el menu: {e}")
            escribir_log(f"ERROR inesperado en el menu: {e}")
            continue
        else:
            intentos_invalidos = 0
        finally:
            pass

# main()  # descomentar para ejecutar el menu interactivo en consola


## 5. Celdas de prueba
A continuación se generan **12 ventas de demostración** (más de las 10 mínimas
solicitadas), se guarda `ventas.csv`, se simula un intento fallido (Reto D),
se calculan métricas con NumPy y se grafica/exporta `ingresos.png` (Reto B).

In [9]:
random.seed(7)
ventas_demo = [
    (1, 2), (2, 5), (3, 12), (4, 1), (5, 1),
    (6, 3), (2, 15), (1, 1), (4, 2), (3, 4),
    (6, 10), (5, 2),
]
fecha_base = datetime(2026, 8, 10, 9, 0, 0)

for i, (pid, cant) in enumerate(ventas_demo):
    registrar_venta(producto_id=pid, cantidad=cant, interactivo=False)

print("\nTotal de ventas en buffer:", len(VENTAS_BUFFER))


 Venta registrada: 2x Laptop -> Total: $1700.00
 Venta registrada: 5x Mouse -> Total: $77.50
 Venta registrada: 12x Teclado -> Total: $285.00 (con descuento 5%)
 Venta registrada: 1x Monitor -> Total: $199.99
 Venta registrada: 1x Silla Gamer -> Total: $120.00
 Venta registrada: 3x Audifonos -> Total: $135.00
 Venta registrada: 15x Mouse -> Total: $220.88 (con descuento 5%)
 Venta registrada: 1x Laptop -> Total: $850.00
 Venta registrada: 2x Monitor -> Total: $399.98
 Venta registrada: 4x Teclado -> Total: $100.00
 Venta registrada: 10x Audifonos -> Total: $427.50 (con descuento 5%)
 Venta registrada: 2x Silla Gamer -> Total: $240.00

Total de ventas en buffer: 12


In [10]:
# Reto D: intento fallido con un producto_id que no existe
registrar_venta(producto_id=99, cantidad=1, interactivo=False)


 El producto no existe en el catalogo.


In [11]:
guardar_ventas_csv()


 12 venta(s) guardadas en /home/claude/minitienda/ventas.csv


In [12]:
calcular_metricas()



--- VENTAS REGISTRADAS (12) ---
 id_venta               fecha  producto_id    producto  cantidad  precio_unitario  descuento   total
        1 2026-08-18 01:02:00            1      Laptop         2           850.00       0.00 1700.00
        2 2026-08-18 01:02:00            2       Mouse         5            15.50       0.00   77.50
        3 2026-08-18 01:02:00            3     Teclado        12            25.00      15.00  285.00
        4 2026-08-18 01:02:00            4     Monitor         1           199.99       0.00  199.99
        5 2026-08-18 01:02:00            5 Silla Gamer         1           120.00       0.00  120.00
        6 2026-08-18 01:02:00            6   Audifonos         3            45.00       0.00  135.00
        7 2026-08-18 01:02:00            2       Mouse        15            15.50      11.62  220.88
        8 2026-08-18 01:02:00            1      Laptop         1           850.00       0.00  850.00
        9 2026-08-18 01:02:00            4     Monitor    

In [13]:
graficar_ingresos(guardar_png=True)



--- VENTAS REGISTRADAS (12) ---
 id_venta               fecha  producto_id    producto  cantidad  precio_unitario  descuento   total
        1 2026-08-18 01:02:00            1      Laptop         2           850.00       0.00 1700.00
        2 2026-08-18 01:02:00            2       Mouse         5            15.50       0.00   77.50
        3 2026-08-18 01:02:00            3     Teclado        12            25.00      15.00  285.00
        4 2026-08-18 01:02:00            4     Monitor         1           199.99       0.00  199.99
        5 2026-08-18 01:02:00            5 Silla Gamer         1           120.00       0.00  120.00
        6 2026-08-18 01:02:00            6   Audifonos         3            45.00       0.00  135.00
        7 2026-08-18 01:02:00            2       Mouse        15            15.50      11.62  220.88
        8 2026-08-18 01:02:00            1      Laptop         1           850.00       0.00  850.00
        9 2026-08-18 01:02:00            4     Monitor    

In [14]:
# Reto A: agregar un producto nuevo y actualizar precio/stock
nuevo_id = agregar_producto("Webcam HD", "Electronica", 55.0, 20)
actualizar_precio_stock(nuevo_id, nuevo_precio=49.99, nuevo_stock=18)
mostrar_catalogo()


 Producto 'Webcam HD' agregado con ID 7.


 Producto actualizado correctamente.

--- CATALOGO DE PRODUCTOS ---
ID  Nombre         Categoria      Precio    Stock 
1   Laptop         Electronica    $850.00   7     
2   Mouse          Electronica    $15.50    30    
3   Teclado        Electronica    $25.00    24    
4   Monitor        Electronica    $199.99   12    
5   Silla Gamer    Muebles        $120.00   5     
6   Audifonos      Electronica    $45.00    12    
7   Webcam HD      Electronica    $49.99    18    


In [15]:
# Verificacion del contenido de log.txt (incluye el intento fallido del Reto D)
with open(LOG_PATH, "r", encoding="utf-8") as f:
    print(f.read())


[2026-08-18 01:02:00] VENTA OK: producto_id=1, cantidad=2, total=1700.0
[2026-08-18 01:02:00] VENTA OK: producto_id=2, cantidad=5, total=77.5
[2026-08-18 01:02:00] VENTA OK: producto_id=3, cantidad=12, total=285.0
[2026-08-18 01:02:00] VENTA OK: producto_id=4, cantidad=1, total=199.99
[2026-08-18 01:02:00] VENTA OK: producto_id=5, cantidad=1, total=120.0
[2026-08-18 01:02:00] VENTA OK: producto_id=6, cantidad=3, total=135.0
[2026-08-18 01:02:00] VENTA OK: producto_id=2, cantidad=15, total=220.88
[2026-08-18 01:02:00] VENTA OK: producto_id=1, cantidad=1, total=850.0
[2026-08-18 01:02:00] VENTA OK: producto_id=4, cantidad=2, total=399.98
[2026-08-18 01:02:00] VENTA OK: producto_id=3, cantidad=4, total=100.0
[2026-08-18 01:02:00] VENTA OK: producto_id=6, cantidad=10, total=427.5
[2026-08-18 01:02:00] VENTA OK: producto_id=5, cantidad=2, total=240.0
[2026-08-18 01:02:00] INTENTO FALLIDO: producto_id=99 no existe en el catalogo.
[2026-08-18 01:02:00] CSV actualizado con 12 venta(s) nuevas.


## 6. Preguntas de la asignación

**¿Qué parte la hizo Pandas? ¿Qué parte NumPy?**
Pandas se usa para estructurar las ventas como `DataFrame` (a partir de la lista de
diccionarios `VENTAS_BUFFER`), para leer/escribir `ventas.csv` (`pd.read_csv`,
`to_csv`) y para agrupar los ingresos por producto con `groupby("producto")["total"].sum()`.
NumPy se usa en `calcular_metricas()` para convertir la columna `total` en un
arreglo (`np.array`) y calcular `sum`, `mean`, `std`, `max` y `min`.

**¿Dónde usaste try/except y por qué?**
En `registrar_venta()` (validar que el ID y la cantidad sean números enteros,
capturar `ValueError`), en `leer_ventas_csv()` (capturar `FileNotFoundError`
cuando `ventas.csv` no existe todavía), en `guardar_ventas_csv()` (capturar
cualquier error al escribir el archivo) y en el cálculo del precio promedio
por unidad, donde se controla una posible `ZeroDivisionError`. También el
menú principal (`main()`) envuelve cada opción en `try/except` para que un
error inesperado no cierre el programa.

**¿Qué estructuras son tuplas, listas y diccionarios en el código?**
- *Tuplas*: cada producto del catálogo, `(id, nombre, categoria)`.
- *Listas*: `CATALOGO` (lista de tuplas), `VENTAS_BUFFER` (lista de diccionarios,
  el buffer de ventas de la sesión) e `IDS_VENDIDOS` (lista/arreglo de ids vendidos).
- *Diccionarios*: `PRECIOS` (id → precio) y `STOCK` (id → cantidad disponible).
